# Olist E-Commerce — Data Cleaning
**Tool:** Python 3.14 / Pandas  
**Input:** 9 raw CSV files in `Data/Raw/`  
**Output:** Cleaned DataFrames exported to `Data/Cleaned/`  

## Objective
Assess, clean, and validate all 9 source tables.
Produce a reliable, analysis-ready dataset with all
decisions documented inline.

## 1. Data Loading & Structural Audit

Load all source tables and assess structural integrity across
every dimension: shape, data types, null distribution, and
duplicate presence. No modifications in this step, observation only.

In [6]:
import pandas as pd
import numpy as np
import os

RAW_PATH = r"C:\Users\USER\OneDrive\Documents\Data Portfolio\Operations_Analytics_Portfolio\01_Olist_Analysis\Data\Raw"

FILES = {
    "Orders":        "olist_orders_dataset.csv",
    "Order_Items":   "olist_order_items_dataset.csv",
    "Payments":      "olist_order_payments_dataset.csv",
    "Reviews":       "olist_order_reviews_dataset.csv",
    "Customers":     "olist_customers_dataset.csv",
    "Products":      "olist_products_dataset.csv",
    "Sellers":       "olist_sellers_dataset.csv",
    "Geolocation":   "olist_geolocation_dataset.csv",
    "Category_Translation": "product_category_name_translation.csv"
}

tables = {}
for name, filename in FILES.items():
    filepath = os.path.join(RAW_PATH, filename)
    tables[name] = pd.read_csv(filepath)
    print(f"Loaded: {name} — {tables[name].shape[0]:,} rows × {tables[name].shape[1]} columns")

Loaded: Orders — 99,441 rows × 8 columns
Loaded: Order_Items — 112,650 rows × 7 columns
Loaded: Payments — 103,886 rows × 5 columns
Loaded: Reviews — 99,224 rows × 7 columns
Loaded: Customers — 99,441 rows × 5 columns
Loaded: Products — 32,951 rows × 9 columns
Loaded: Sellers — 3,095 rows × 4 columns
Loaded: Geolocation — 1,000,163 rows × 5 columns
Loaded: Category_Translation — 71 rows × 2 columns


In [ ]:
#  Structural Audit ───────────────────────────────────────────
# For every table: shape, dtypes, null counts, null %, duplicates

audit_results = []

for name, df in tables.items():
    total_rows = len(df)
    total_nulls = df.isnull().sum().sum()
    null_pct = round((total_nulls / (total_rows * len(df.columns))) * 100, 2)
    duplicates = df.duplicated().sum()
    
    audit_results.append({
        "Table": name,
        "Rows": total_rows,
        "Columns": len(df.columns),
        "Total_Nulls": total_nulls,
        "Null_%": null_pct,
        "Duplicate_Rows": duplicates
    })

audit_df = pd.DataFrame(audit_results)
print("═" * 70)
print("STRUCTURAL AUDIT SUMMARY")
print("═" * 70)
print(audit_df.to_string(index=False))

══════════════════════════════════════════════════════════════════════
STRUCTURAL AUDIT SUMMARY
══════════════════════════════════════════════════════════════════════
               Table    Rows  Columns  Total_Nulls  Null_%  Duplicate_Rows
              Orders   99441        8         4908    0.62               0
         Order_Items  112650        7            0    0.00               0
            Payments  103886        5            0    0.00               0
             Reviews   99224        7       145903   21.01               0
           Customers   99441        5            0    0.00               0
            Products   32951        9         2448    0.83               0
             Sellers    3095        4            0    0.00               0
         Geolocation 1000163        5            0    0.00          261831
Category_Translation      71        2            0    0.00               0


: 

In [ ]:
#  Column-Level Null Audit ────────────────────────────────────
# Identifies exactly which columns carry nulls and how severe

print("═" * 70)
print("COLUMN-LEVEL NULL AUDIT: Tables with nulls only")
print("═" * 70)

for name, df in tables.items():
    null_cols = df.isnull().sum()
    null_cols = null_cols[null_cols > 0]
    
    if len(null_cols) > 0:
        print(f"\n| {name} |")
        for col, count in null_cols.items():
            pct = round((count / len(df)) * 100, 2)
            print(f"   {col:<45} {count:>6} nulls  ({pct}%)")

══════════════════════════════════════════════════════════════════════
COLUMN-LEVEL NULL AUDIT: Tables with nulls only
══════════════════════════════════════════════════════════════════════

| Orders |
   order_approved_at                                160 nulls  (0.16%)
   order_delivered_carrier_date                    1783 nulls  (1.79%)
   order_delivered_customer_date                   2965 nulls  (2.98%)

| Reviews |
   review_comment_title                           87656 nulls  (88.34%)
   review_comment_message                         58247 nulls  (58.7%)

| Products |
   product_category_name                            610 nulls  (1.85%)
   product_name_lenght                              610 nulls  (1.85%)
   product_description_lenght                       610 nulls  (1.85%)
   product_photos_qty                               610 nulls  (1.85%)
   product_weight_g                                   2 nulls  (0.01%)
   product_length_cm                                  2 null

## 2. Table-Level Cleaning

Resolves all structural issues identified in the audit.
Cleaning sequence follows dependency order, tables referenced
by joins are cleaned before the tables that depend on them.
Every decision is documented inline with before/after counts.

In [ ]:
#  ORDERS: Timestamp Null Handling ──────────────────────────
#
# Three timestamp columns carry nulls:
#   order_approved_at        160 nulls (0.16%) — payment not confirmed
#   order_delivered_carrier_date  1783 nulls (1.79%) — never dispatched
#   order_delivered_customer_date 2965 nulls (2.98%) — never delivered
#
# These nulls represent real operational states, not data entry errors.
# Dropping these rows would remove cancelled, pending, and undelivered
# orders from the analysis — distorting fulfilment and cancellation metrics.
# Resolution: Retain all rows. Null timestamps are flagged via
# a derived order_status field already present in the table.
# ──────────────────────────────────────────────────────────────

orders = tables["Orders"].copy()

# Convert all timestamp columns to datetime dtype
# pandas stores dates as generic objects by default — 
# converting to datetime enables date arithmetic (e.g. delivery time calculation)
timestamp_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in timestamp_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

# Verify dtypes converted correctly
print("| Orders: Timestamp dtype verification |")
print(orders[timestamp_cols].dtypes)
print(f"\nRow count before: {len(tables['Orders']):,}")
print(f"Row count after:  {len(orders):,}")
print(f"Nulls retained (intentional): order_approved_at={orders['order_approved_at'].isnull().sum()}, "
      f"carrier_date={orders['order_delivered_carrier_date'].isnull().sum()}, "
      f"customer_date={orders['order_delivered_customer_date'].isnull().sum()}")

| Orders: Timestamp dtype verification |
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

Row count before: 99,441
Row count after:  99,441
Nulls retained (intentional): order_approved_at=160, carrier_date=1783, customer_date=2965


In [ ]:
#  REVIEWS: Free-Text Null Handling ─────────────────────────
#
# review_comment_title:   87,656 nulls (88.34%)
# review_comment_message: 58,247 nulls (58.70%)
#
# Nulls reflect customers who submitted a star rating without
# writing a comment — a standard review behaviour pattern.
# These are valid empty responses, not missing data.
# Resolution: Fill with empty string so downstream text operations
# do not error on NaN. Star rating (review_score) is intact and
# carries full analytical value for satisfaction analysis.
# ──────────────────────────────────────────────────────────────

reviews = tables["Reviews"].copy()

before_nulls = reviews[["review_comment_title", "review_comment_message"]].isnull().sum()

reviews["review_comment_title"] = reviews["review_comment_title"].fillna("")
reviews["review_comment_message"] = reviews["review_comment_message"].fillna("")

# Convert review dates to datetime
reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"], errors="coerce")
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"], errors="coerce")

after_nulls = reviews[["review_comment_title", "review_comment_message"]].isnull().sum()

print("| Reviews: Free-text null resolution |")
print(f"review_comment_title  — Before: {before_nulls['review_comment_title']:,}  After: {after_nulls['review_comment_title']}")
print(f"review_comment_message — Before: {before_nulls['review_comment_message']:,}  After: {after_nulls['review_comment_message']}")
print(f"\nRow count unchanged: {len(reviews):,}")
print(f"review_score nulls: {reviews['review_score'].isnull().sum()} — analytical column intact")

| Reviews: Free-text null resolution |
review_comment_title  — Before: 87,656  After: 0
review_comment_message — Before: 58,247  After: 0

Row count unchanged: 99,224
review_score nulls: 0 — analytical column intact


In [ ]:
#  PRODUCTS: Orphaned Records & Dimension Nulls ─────────────
#
# 610 rows missing product_category_name, product_name_lenght,
# product_description_lenght, and product_photos_qty simultaneously.
# Identical null pattern across all four fields = orphaned product
# records with no usable metadata.
#
# Resolution: Assign category "unknown" rather than dropping.
# These products appear in order_items and represent real revenue.
# Dropping them would create unmatched keys in the master join
# and silently exclude transactions from revenue totals.
#
# 2 rows missing physical dimensions (weight, length, height, width).
# Resolution: Impute with column median — physical dimensions follow
# a distribution; median is robust to outliers and preserves row count.
# ──────────────────────────────────────────────────────────────

products = tables["Products"].copy()

before_category_nulls = products["product_category_name"].isnull().sum()

# Fill category nulls with "unknown"
products["product_category_name"] = products["product_category_name"].fillna("unknown")

# Impute physical dimension nulls with column median
dimension_cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
for col in dimension_cols:
    median_val = products[col].median()
    null_count = products[col].isnull().sum()
    products[col] = products[col].fillna(median_val)
    print(f"{col}: {null_count} null(s) imputed with median ({median_val})")

# Rename misspelled columns — source data contains typos
# product_name_lenght → product_name_length (typo in source)
# product_description_lenght → product_description_length (typo in source)
products = products.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length"
})

after_category_nulls = products["product_category_name"].isnull().sum()

print(f"\n| Products: Cleaning summary |")
print(f"Category nulls  — Before: {before_category_nulls}  After: {after_category_nulls} (filled: 'unknown')")
print(f"Column renames  — product_name_lenght → product_name_length")
print(f"                  product_description_lenght → product_description_length")
print(f"Row count unchanged: {len(products):,}")

product_weight_g: 2 null(s) imputed with median (700.0)
product_length_cm: 2 null(s) imputed with median (25.0)
product_height_cm: 2 null(s) imputed with median (13.0)
product_width_cm: 2 null(s) imputed with median (20.0)

| Products: Cleaning summary |
Category nulls  — Before: 610  After: 0 (filled: 'unknown')
Column renames  — product_name_lenght → product_name_length
                  product_description_lenght → product_description_length
Row count unchanged: 32,951


In [ ]:
#  GEOLOCATION: Deduplication ───────────────────────────────
#
# 261,831 duplicate rows (26.1% of table) — multiple coordinate
# entries per zip code prefix from different source records.
#
# This table is used exclusively for state-level regional mapping.
# We do not need coordinate precision below state level for this analysis.
# Resolution: Aggregate to state level — one representative lat/lng
# per state using the median coordinate across all zip prefixes
# in that state. Median chosen over mean to resist outlier coordinates.
#
# Geolocation joins to customers and sellers via zip code prefix.
# We retain zip-level deduplication as a fallback join key.
# ──────────────────────────────────────────────────────────────

geo = tables["Geolocation"].copy()

before_rows = len(geo)

# Deduplicate on zip code prefix — keep first representative coordinate
geo_zip = geo.drop_duplicates(subset=["geolocation_zip_code_prefix"], keep="first")

# Build state-level aggregation for mapping use
geo_state = geo.groupby("geolocation_state").agg(
    lat=("geolocation_lat", "median"),
    lng=("geolocation_lng", "median")
).reset_index()

after_rows = len(geo_zip)
removed = before_rows - after_rows

print(" | Geolocation: Deduplication summary |")
print(f"Rows before: {before_rows:,}")
print(f"Rows after (zip-level):  {after_rows:,}")
print(f"Duplicates removed: {removed:,} ({round(removed/before_rows*100,1)}%)")
print(f"\nState-level aggregation: {len(geo_state)} states for regional mapping")
print(geo_state.sort_values("geolocation_state").to_string(index=False))

 | Geolocation: Deduplication summary |
Rows before: 1,000,163
Rows after (zip-level):  19,015
Duplicates removed: 981,148 (98.1%)

State-level aggregation: 27 states for regional mapping
geolocation_state        lat        lng
               AC  -9.962046 -67.830638
               AL  -9.636914 -35.754053
               AM  -3.089704 -60.011998
               AP   0.031212 -51.074329
               BA -12.956328 -38.957697
               CE  -3.789734 -38.585162
               DF -15.817696 -47.939322
               ES -20.294495 -40.320785
               GO -16.682929 -49.265681
               MA  -3.288063 -44.298288
               MG -19.921031 -43.979365
               MS -20.482335 -54.614251
               MT -15.244524 -56.039189
               PA  -1.462833 -48.484188
               PB  -7.125067 -35.488819
               PE  -8.081400 -34.953305
               PI  -5.099470 -42.763117
               PR -25.384058 -50.618097
               RJ -22.874466 -43.242050
            

In [21]:
#  ORDER_ITEMS, PAYMENTS, CUSTOMERS, SELLERS ─────────────────
#
# Audit confirmed zero nulls and zero duplicates across all four tables.
# Cleaning limited to dtype corrections and column standardisation.
# ──────────────────────────────────────────────────────────────

order_items = tables["Order_Items"].copy()
payments    = tables["Payments"].copy()
customers   = tables["Customers"].copy()
sellers     = tables["Sellers"].copy()

# Order_Items: shipping_limit_date to datetime
order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"], errors="coerce"
)

# Payments: payment_value is already float — confirm
print("| Payments: payment_value dtype |")
print(f"   {payments['payment_value'].dtype}")

# Customers: standardise state column name for join consistency
customers = customers.rename(columns={
    "customer_state": "state",
    "customer_city": "city"
})

# Sellers: standardise state column name for join consistency
sellers = sellers.rename(columns={
    "seller_state": "state",
    "seller_city": "city"
})

print("\n| Remaining tables: Cleaning summary |")
print(f"Order_Items: shipping_limit_date converted to datetime, {len(order_items):,} rows intact")
print(f"Payments: no changes required, {len(payments):,} rows intact")
print(f"Customers: state/city columns standardised, {len(customers):,} rows intact")
print(f"Sellers: state/city columns standardised, {len(sellers):,} rows intact")

| Payments: payment_value dtype |
   float64

| Remaining tables: Cleaning summary |
Order_Items: shipping_limit_date converted to datetime, 112,650 rows intact
Payments: no changes required, 103,886 rows intact
Customers: state/city columns standardised, 99,441 rows intact
Sellers: state/city columns standardised, 3,095 rows intact


## 3. Category Translation & Master Table Construction

Translates product categories from Portuguese to English.
Joins all cleaned tables into a single analytical master table.
This table is the authoritative source for all subsequent analysis.

In [ ]:
#  CATEGORY TRANSLATION ───────────────────────────────────────
#
# Product categories in the source data are in Portuguese.
# The translation table maps 71 categories to English equivalents.
# Products assigned "unknown" in Step 2 (610 records) retain that
# label — no English equivalent exists for an uncategorised product.
# ──────────────────────────────────────────────────────────────

translation = tables["Category_Translation"].copy()

# Rename columns for clarity before merging
translation = translation.rename(columns={
    "product_category_name":            "product_category_name",
    "product_category_name_english":    "category_english"
})

# Merge translation into products table
# Left join — retains all products including the 610 "unknown" records
# that have no Portuguese name and therefore no English equivalent
products_translated = products.merge(
    translation,
    on="product_category_name",
    how="left"
)

# For the 610 "unknown" records, category_english will be null after join
# Fill with "Unknown" for consistency
products_translated["category_english"] = (
    products_translated["category_english"]
    .fillna("Unknown")
)

# Verify translation coverage
total_products    = len(products_translated)
translated        = products_translated["category_english"].ne("Unknown").sum()
untranslated      = products_translated["category_english"].eq("Unknown").sum()

print("| Category translation summary |")
print(f"Total products:       {total_products:,}")
print(f"Successfully mapped:  {translated:,} ({round(translated/total_products*100,1)}%)")
print(f"Retained as Unknown:  {untranslated:,} ({round(untranslated/total_products*100,1)}%)")
print(f"\nSample — unique English categories: {products_translated['category_english'].nunique()}")
print(sorted(products_translated["category_english"].dropna().unique())[:15])

| Category translation summary |
Total products:       32,951
Successfully mapped:  32,328 (98.1%)
Retained as Unknown:  623 (1.9%)

Sample — unique English categories: 72
['Unknown', 'agro_industry_and_commerce', 'air_conditioning', 'art', 'arts_and_craftmanship', 'audio', 'auto', 'baby', 'bed_bath_table', 'books_general_interest', 'books_imported', 'books_technical', 'cds_dvds_musicals', 'christmas_supplies', 'cine_photo']


In [ ]:
#  MASTER ANALYTICAL TABLE ────────────────────────────────────
#
# Joins all cleaned tables into one flat analytical dataset.
# Join sequence follows the relational spine of the schema:
#
#   orders (spine)
#     → order_items     on order_id          [1:many — one order, many items]
#     → products        on product_id        [many:1 — many items, one product]
#     → customers       on customer_id       [many:1 — many orders, one customer]
#     → payments        on order_id          [1:many — aggregated before join]
#     → reviews         on order_id          [1:1 — one review per order]
#     → sellers         on seller_id         [many:1 — many items, one seller]
#
# Payments are aggregated to order level before joining because
# one order can have multiple payment rows (e.g. voucher + card).
# Joining un-aggregated payments would inflate row count and
# double-count revenue — a critical data integrity error.
# ──────────────────────────────────────────────────────────────

# Step A — Aggregate payments to order level before joining
payments_agg = payments.groupby("order_id").agg(
    total_payment_value=("payment_value", "sum"),
    payment_installments=("payment_installments", "max"),
    payment_type=("payment_type", "first")          # primary payment method
).reset_index()

print(f"Payments aggregated: {len(payments):,} rows → {len(payments_agg):,} order-level rows")

# Step B — Build the master table join sequence
master = (
    orders
    .merge(order_items,          on="order_id",    how="left")
    .merge(products_translated[["product_id",
                                 "product_category_name",
                                 "category_english",
                                 "product_weight_g",
                                 "product_length_cm",
                                 "product_height_cm",
                                 "product_width_cm"]],
                                 on="product_id",  how="left")
    .merge(customers,            on="customer_id", how="left")
    .merge(payments_agg,         on="order_id",    how="left")
    .merge(reviews[["order_id",
                    "review_score",
                    "review_comment_message"]],
                                 on="order_id",    how="left")
    .merge(sellers[["seller_id",
                    "state",
                    "city"]].rename(columns={
                        "state": "seller_state",
                        "city":  "seller_city"}),
                                 on="seller_id",   how="left")
)

print(f"\nMaster table shape: {master.shape[0]:,} rows × {master.shape[1]} columns")
print(f"\nColumns in master table:")
for col in master.columns:
    print(f"   {col}")

Payments aggregated: 103,886 rows → 99,440 order-level rows

Master table shape: 114,092 rows × 31 columns

Columns in master table:
   order_id
   customer_id
   order_status
   order_purchase_timestamp
   order_approved_at
   order_delivered_carrier_date
   order_delivered_customer_date
   order_estimated_delivery_date
   order_item_id
   product_id
   seller_id
   shipping_limit_date
   price
   freight_value
   product_category_name
   category_english
   product_weight_g
   product_length_cm
   product_height_cm
   product_width_cm
   customer_unique_id
   customer_zip_code_prefix
   city
   state
   total_payment_value
   payment_installments
   payment_type
   review_score
   review_comment_message
   seller_state
   seller_city


In [ ]:
#  MASTER TABLE VALIDATION ────────────────────────────────────
#
# Confirms join integrity before exporting.
# Checks: row count logic, null distribution, revenue totals,
# and referential integrity on key join columns.
# ──────────────────────────────────────────────────────────────

print("═" * 65)
print("MASTER TABLE VALIDATION")
print("═" * 65)

# 1. Row count
# Master row count should equal order_items row count (112,650)
# because orders expand to item-level on the order_items join.
# Each item in an order becomes its own row in the master table.
print(f"\n1. Row count")
print(f"   Expected (order_items rows): 112,650")
print(f"   Actual:                      {len(master):,}")
print(f"   Match: {len(master) == 112650}")

# 2. Revenue integrity
# Sum of (price + freight_value) in master should equal
# sum of the same in source order_items — no inflation from joins
master_revenue    = (master["price"] + master["freight_value"]).sum()
source_revenue    = (order_items["price"] + order_items["freight_value"]).sum()
print(f"\n2. Revenue integrity")
print(f"   Source order_items total:  R$ {source_revenue:,.2f}")
print(f"   Master table total:        R$ {master_revenue:,.2f}")
print(f"   Variance:                  R$ {abs(master_revenue - source_revenue):,.2f}")
print(f"   Match: {round(master_revenue, 2) == round(source_revenue, 2)}")

# 3. Null audit on critical analytical columns
critical_cols = [
    "order_id", "customer_id", "order_status",
    "order_purchase_timestamp", "price", "freight_value",
    "state", "category_english", "review_score"
]
print(f"\n3. Null audit — critical columns")
for col in critical_cols:
    nulls = master[col].isnull().sum()
    pct   = round(nulls / len(master) * 100, 2)
    flag  = " ⚠" if nulls > 0 else ""
    print(f"   {col:<35} {nulls:>6} nulls ({pct}%){flag}")

# 4. Key value ranges — sanity check
print(f"\n4. Value range checks")
print(f"   Price range:         R$ {master['price'].min():.2f} — R$ {master['price'].max():,.2f}")
print(f"   Freight range:       R$ {master['freight_value'].min():.2f} — R$ {master['freight_value'].max():,.2f}")
print(f"   Review score range:  {master['review_score'].min()} — {master['review_score'].max()}")
print(f"   Order date range:    {master['order_purchase_timestamp'].min().date()} — {master['order_purchase_timestamp'].max().date()}")
print(f"   Unique orders:       {master['order_id'].nunique():,}")
print(f"   Unique customers:    {master['customer_id'].nunique():,}")
print(f"   Unique sellers:      {master['seller_id'].nunique():,}")
print(f"   Brazilian states:    {master['state'].nunique()}")

═════════════════════════════════════════════════════════════════
MASTER TABLE VALIDATION
═════════════════════════════════════════════════════════════════

1. Row count
   Expected (order_items rows): 112,650
   Actual:                      114,092
   Match: False

2. Revenue integrity
   Source order_items total:  R$ 15,843,553.24
   Master table total:        R$ 15,915,872.32
   Variance:                  R$ 72,319.08
   Match: False

3. Null audit — critical columns
   order_id                                 0 nulls (0.0%)
   customer_id                              0 nulls (0.0%)
   order_status                             0 nulls (0.0%)
   order_purchase_timestamp                 0 nulls (0.0%)
   price                                  778 nulls (0.68%) ⚠
   freight_value                          778 nulls (0.68%) ⚠
   state                                    0 nulls (0.0%)
   category_english                       778 nulls (0.68%) ⚠
   review_score                           96

In [26]:
# Diagnose: duplicate order_ids in reviews
review_dupes = reviews.groupby("order_id").size()
review_dupes = review_dupes[review_dupes > 1]
print(f"Orders with multiple review records: {len(review_dupes):,}")
print(f"Extra rows this creates: {(review_dupes - 1).sum():,}")
print(f"\nSample duplicate review orders:")
print(review_dupes.head(10))

Orders with multiple review records: 547
Extra rows this creates: 551

Sample duplicate review orders:
order_id
0035246a40f520710769010f752e7507    2
013056cfe49763c6f66bda03396c5ee3    2
0176a6846bcb3b0d3aa3116a9a768597    2
02355020fd0a40a0d56df9f6ff060413    2
029863af4b968de1e5d6a82782e662f5    2
02e0b68852217f5715fb9cc885829454    2
02e723e8edb4a123d414f56cc9c4665e    2
03515a836bb855b03f7df9dee520a8fc    2
03c939fd7fd3b38f8485a0f95798f1f6    3
03eba6d9fef8f5b3e811d4b5a7cca9cd    2
dtype: int64


In [27]:
# Diagnose: orders with no matching order_items
orders_no_items = master[master["price"].isnull()]["order_id"].nunique()
status_breakdown = master[master["price"].isnull()]["order_status"].value_counts()
print(f"Orders with no item records: {orders_no_items:,}")
print(f"\nOrder status breakdown for these orders:")
print(status_breakdown)

Orders with no item records: 775

Order status breakdown for these orders:
order_status
unavailable    605
canceled       165
created          5
invoiced         2
shipped          1
Name: count, dtype: int64


In [28]:
# Diagnose: orders with no review record
no_review_orders = master[master["review_score"].isnull()]["order_id"].nunique()
print(f"Orders with no review record: {no_review_orders:,}")

Orders with no review record: 768


## 3.1 Join Integrity Fixes

Resolves three structural issues surfaced during master table validation:
duplicate review records inflating row count, orders with no item records,
and orders with no review. Each fix is surgical — no rows deleted
without documented justification.

In [29]:
#  FIX: DUPLICATE REVIEW RECORDS ─────────────────────────────
#
# 547 orders carry more than one review record in the source table,
# generating 551 extra rows after joining. Root cause: customers
# were permitted to revise their review, creating a second record
# rather than updating the original.
#
# Resolution: Retain the most recent review per order, identified
# by review_answer_timestamp. Most recent review reflects the
# customer's final settled sentiment — analytically more reliable
# than the first submission.
#
# Before: 99,224 review rows
# After:  expected 98,673 review rows (99,224 - 551)
# ──────────────────────────────────────────────────────────────

before_review_rows = len(reviews)

reviews_deduped = (
    reviews
    .sort_values("review_answer_timestamp", ascending=False)
    .drop_duplicates(subset="order_id", keep="first")
    .reset_index(drop=True)
)

after_review_rows = len(reviews_deduped)
removed = before_review_rows - after_review_rows

print("| Review deduplication |")
print(f"Rows before: {before_review_rows:,}")
print(f"Rows after:  {after_review_rows:,}")
print(f"Removed:     {removed:,} duplicate review records")
print(f"Expected:    551 — Match: {removed == 551}")

| Review deduplication |
Rows before: 99,224
Rows after:  98,673
Removed:     551 duplicate review records
Expected:    551 — Match: True


In [30]:
#  REBUILD MASTER TABLE ───────────────────────────────────────
#
# Rebuilds master table using reviews_deduped in place of reviews.
# All other joins identical to initial build.
# Expected row count after rebuild: 112,650 (order_items spine).
# ──────────────────────────────────────────────────────────────

master = (
    orders
    .merge(order_items,          on="order_id",    how="left")
    .merge(products_translated[["product_id",
                                 "product_category_name",
                                 "category_english",
                                 "product_weight_g",
                                 "product_length_cm",
                                 "product_height_cm",
                                 "product_width_cm"]],
                                 on="product_id",  how="left")
    .merge(customers,            on="customer_id", how="left")
    .merge(payments_agg,         on="order_id",    how="left")
    .merge(reviews_deduped[["order_id",
                             "review_score",
                             "review_comment_message"]],
                                 on="order_id",    how="left")
    .merge(sellers[["seller_id",
                    "state",
                    "city"]].rename(columns={
                        "state": "seller_state",
                        "city":  "seller_city"}),
                                 on="seller_id",   how="left")
)

print(f"Master table rebuilt: {master.shape[0]:,} rows × {master.shape[1]} columns")

Master table rebuilt: 113,425 rows × 31 columns


In [33]:
#  VALIDATION: REBUILT MASTER TABLE ──────────────────────────
#
# Repeats all validation checks against the rebuilt master.
# All four checks must pass before export proceeds.
# ──────────────────────────────────────────────────────────────

print("═" * 65)
print("MASTER TABLE VALIDATION; POST-FIX")
print("═" * 65)

# 1. Row count
print(f"\n1. Row count")
print(f"   Expected: 112,650")
print(f"   Actual:   {len(master):,}")
print(f"   Match: {len(master) == 112650}")

# 2. Revenue integrity
master_revenue = (master["price"].fillna(0) + master["freight_value"].fillna(0)).sum()
source_revenue = (order_items["price"] + order_items["freight_value"]).sum()
variance       = abs(master_revenue - source_revenue)
print(f"\n2. Revenue integrity")
print(f"   Source total:  R$ {source_revenue:,.2f}")
print(f"   Master total:  R$ {master_revenue:,.2f}")
print(f"   Variance:      R$ {variance:,.2f}")
print(f"   Match: {round(variance, 2) == 0.00}")

# 3. Null audit
critical_cols = [
    "order_id", "customer_id", "order_status",
    "order_purchase_timestamp", "price", "freight_value",
    "state", "category_english", "review_score"
]
print(f"\n3. Null audit: critical columns")
for col in critical_cols:
    nulls = master[col].isnull().sum()
    pct   = round(nulls / len(master) * 100, 2)
    flag  = " ⚠" if nulls > 0 else " ✓"
    print(f"   {col:<35} {nulls:>6} nulls ({pct}%){flag}")

# 4. Sanity checks
print(f"\n4. Value range checks")
print(f"   Price range:        R$ {master['price'].min():.2f} — R$ {master['price'].max():,.2f}")
print(f"   Freight range:      R$ {master['freight_value'].min():.2f} — R$ {master['freight_value'].max():,.2f}")
print(f"   Review score range: {master['review_score'].min()} — {master['review_score'].max()}")
print(f"   Order date range:   {master['order_purchase_timestamp'].min().date()} — "
      f"{master['order_purchase_timestamp'].max().date()}")
print(f"   Unique orders:      {master['order_id'].nunique():,}")
print(f"   Unique customers:   {master['customer_id'].nunique():,}")
print(f"   Unique sellers:     {master['seller_id'].nunique():,}")
print(f"   States covered:     {master['state'].nunique()}")

# 5. Known retained nulls — documented and expected
print(f"\n5. Retained nulls: documented and expected")
print(f"   price/freight nulls (no-item orders):  "
      f"{master['price'].isnull().sum():,} operational events, not transactions")
print(f"   review_score nulls (unrated orders):   "
      f"{master['review_score'].isnull().sum():,} valid non-response")
print(f"   category_english nulls:                "
      f"{master['category_english'].isnull().sum():,} unmatched product records")

═════════════════════════════════════════════════════════════════
MASTER TABLE VALIDATION; POST-FIX
═════════════════════════════════════════════════════════════════

1. Row count
   Expected: 112,650
   Actual:   113,425
   Match: False

2. Revenue integrity
   Source total:  R$ 15,843,553.24
   Master total:  R$ 15,843,553.24
   Variance:      R$ 0.00
   Match: True

3. Null audit: critical columns
   order_id                                 0 nulls (0.0%) ✓
   customer_id                              0 nulls (0.0%) ✓
   order_status                             0 nulls (0.0%) ✓
   order_purchase_timestamp                 0 nulls (0.0%) ✓
   price                                  775 nulls (0.68%) ⚠
   freight_value                          775 nulls (0.68%) ⚠
   state                                    0 nulls (0.0%) ✓
   category_english                       775 nulls (0.68%) ⚠
   review_score                           961 nulls (0.85%) ⚠

4. Value range checks
   Price range:     

In [ ]:
#  FIX: JOIN SPINE CORRECTION ─────────────────────────────────
#
# Previous build used orders as the left spine, left-joining
# order_items onto it. This retained 775 orders with no item
# records as null-filled rows, inflating master row count to
# 113,425 against the expected 112,650.
#
# These 775 orders carry status: unavailable (605), cancelled
# (165), created (5). They represent orders that entered the
# system but never progressed to item fulfilment. No transaction
# occurred — no price, no product, no seller.
#
# Resolution: Rebuild with order_items as the left spine.
# Orders join right onto items — only orders with confirmed
# items enter the master table. The 775 no-item orders are
# excluded from the item-level master and will be analysed
# separately at order level for cancellation/unavailability metrics.
#
# Before: 113,425 rows (775 phantom rows from no-item orders)
# After:  112,650 rows (one row per confirmed order item)
# ──────────────────────────────────────────────────────────────

# Preserve the 775 no-item orders separately before rebuilding
# These will feed the order-status analysis in Phase 2
orders_no_items = orders[~orders["order_id"].isin(order_items["order_id"])].copy()

print(f"No-item orders preserved separately: {len(orders_no_items):,}")
print(f"Status breakdown:")
print(orders_no_items["order_status"].value_counts().to_string())

# Rebuild master — order_items as the left spine
master = (
    order_items
    .merge(orders,               on="order_id",    how="left")
    .merge(products_translated[["product_id",
                                 "product_category_name",
                                 "category_english",
                                 "product_weight_g",
                                 "product_length_cm",
                                 "product_height_cm",
                                 "product_width_cm"]],
                                 on="product_id",  how="left")
    .merge(customers,            on="customer_id", how="left")
    .merge(payments_agg,         on="order_id",    how="left")
    .merge(reviews_deduped[["order_id",
                             "review_score",
                             "review_comment_message"]],
                                 on="order_id",    how="left")
    .merge(sellers[["seller_id",
                    "state",
                    "city"]].rename(columns={
                        "state": "seller_state",
                        "city":  "seller_city"}),
                                 on="seller_id",   how="left")
)

print(f"\nMaster table rebuilt: {master.shape[0]:,} rows × {master.shape[1]} columns")

No-item orders preserved separately: 775
Status breakdown:
order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1

Master table rebuilt: 112,650 rows × 31 columns


In [36]:
#  FINAL VALIDATION ───────────────────────────────────────────

print("═" * 65)
print("MASTER TABLE: FINAL VALIDATION")
print("═" * 65)

# 1. Row count
print(f"\n1. Row count")
print(f"   Expected: 112,650")
print(f"   Actual:   {len(master):,}")
print(f"   Match: {len(master) == 112650}")

# 2. Revenue integrity
master_revenue = (master["price"] + master["freight_value"]).sum()
source_revenue = (order_items["price"] + order_items["freight_value"]).sum()
variance       = abs(master_revenue - source_revenue)
print(f"\n2. Revenue integrity")
print(f"   Source total:  R$ {source_revenue:,.2f}")
print(f"   Master total:  R$ {master_revenue:,.2f}")
print(f"   Variance:      R$ 0.00 Match: {round(variance, 2) == 0.00}")

# 3. Null audit
critical_cols = [
    "order_id", "customer_id", "order_status",
    "order_purchase_timestamp", "price", "freight_value",
    "state", "category_english", "review_score"
]
print(f"\n3. Null audit: critical columns")
for col in critical_cols:
    nulls = master[col].isnull().sum()
    pct   = round(nulls / len(master) * 100, 2)
    flag  = " ⚠" if nulls > 0 else " ✓"
    print(f"   {col:<35} {nulls:>6} nulls ({pct}%){flag}")

# 4. Sanity checks
print(f"\n4. Value range checks")
print(f"   Price range:        R$ {master['price'].min():.2f} R$ {master['price'].max():,.2f}")
print(f"   Freight range:      R$ {master['freight_value'].min():.2f} R$ {master['freight_value'].max():,.2f}")
print(f"   Review score range: {master['review_score'].min()} {master['review_score'].max()}")
print(f"   Order date range:   {master['order_purchase_timestamp'].min().date()} "
      f"{master['order_purchase_timestamp'].max().date()}")
print(f"   Unique orders:      {master['order_id'].nunique():,}")
print(f"   Unique customers:   {master['customer_id'].nunique():,}")
print(f"   Unique sellers:     {master['seller_id'].nunique():,}")
print(f"   States covered:     {master['state'].nunique()}")

# 5. Retained nulls — expected and documented
print(f"\n5. Retained nulls: expected")
print(f"   review_score:     {master['review_score'].isnull().sum():,} orders with no customer rating")
print(f"   category_english: {master['category_english'].isnull().sum():,} products with no category metadata")

═════════════════════════════════════════════════════════════════
MASTER TABLE: FINAL VALIDATION
═════════════════════════════════════════════════════════════════

1. Row count
   Expected: 112,650
   Actual:   112,650
   Match: True

2. Revenue integrity
   Source total:  R$ 15,843,553.24
   Master total:  R$ 15,843,553.24
   Variance:      R$ 0.00 Match: True

3. Null audit: critical columns
   order_id                                 0 nulls (0.0%) ✓
   customer_id                              0 nulls (0.0%) ✓
   order_status                             0 nulls (0.0%) ✓
   order_purchase_timestamp                 0 nulls (0.0%) ✓
   price                                    0 nulls (0.0%) ✓
   freight_value                            0 nulls (0.0%) ✓
   state                                    0 nulls (0.0%) ✓
   category_english                         0 nulls (0.0%) ✓
   review_score                           942 nulls (0.84%) ⚠

4. Value range checks
   Price range:        R$ 0.85

## 4. Export

Exports the validated master analytical table and the preserved
no-item orders table to Data/Cleaned/. These files are the
authoritative source for all downstream analysis.

In [ ]:
#  EXPORT CLEANED TABLES ──────────────────────────────────────
#
# Two files exported:
#
# 1. Olist_Master.csv: 112,650 rows, 31 columns
#    Item-level analytical table. One row per confirmed order item.
#    Source for all revenue, category, regional, and LTV analysis.
#
# 2. Olist_Orders_No_Items.csv: 775 rows, 8 columns
#    Order-level records with no item fulfilment.
#    Source for cancellation rate and unavailability analysis.
#    Excluded from revenue analysis by design.
#
# Index excluded from export: row numbers carry no analytical value
# and add noise when the file is loaded in SQL or Power BI.
# ──────────────────────────────────────────────────────────────

import os

CLEANED_PATH = r"C:\Users\USER\OneDrive\Documents\Data Portfolio\Operations_Analytics_Portfolio\01_Olist_Analysis\Data\Cleaned"

os.makedirs(CLEANED_PATH, exist_ok=True)

# Export master table
master_path = os.path.join(CLEANED_PATH, "Olist_Master.csv")
master.to_csv(master_path, index=False)
master_size = os.path.getsize(master_path) / (1024 * 1024)
print(f"Exported: Olist_Master.csv")
print(f"   Rows:    {len(master):,}")
print(f"   Columns: {master.shape[1]}")
print(f"   Size:    {master_size:.1f} MB")

# Export no-item orders
no_items_path = os.path.join(CLEANED_PATH, "Olist_Orders_No_Items.csv")
orders_no_items.to_csv(no_items_path, index=False)
no_items_size = os.path.getsize(no_items_path) / (1024 * 1024)
print(f"\nExported: Olist_Orders_No_Items.csv")
print(f"   Rows:    {len(orders_no_items):,}")
print(f"   Columns: {orders_no_items.shape[1]}")
print(f"   Size:    {no_items_size:.1f} MB")

print(f"\nAll exports written to: {CLEANED_PATH}")

Exported: Olist_Master.csv
   Rows:    112,650
   Columns: 31
   Size:    47.4 MB

Exported: Olist_Orders_No_Items.csv
   Rows:    775
   Columns: 8
   Size:    0.1 MB

All exports written to: C:\Users\USER\OneDrive\Documents\Data Portfolio\Operations_Analytics_Portfolio\01_Olist_Analysis\Data\Cleaned
